# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"rows: {len(df)}")
print(f"total revenue: ${total_revenue:,.2f}")
print(f"total units: {total_units}")

rows: 400
total revenue: $8,520.00
total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
by_category = (df.groupby('category', as_index=False)['revenue']
               .sum()
               .sort_values('revenue', ascending=False))
by_category['share_pct'] = (by_category['revenue'] / df['revenue'].sum() * 100).round(1)
by_category

,category,revenue,share_pct
1,Food,4293.0,50.4
2,Merch,1771.5,20.8
0,Drink,1554.0,18.2
3,RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
by_vendor = (df.groupby('vendor_id')
             .agg(avg_revenue=('revenue', 'mean'), order_count=('revenue', 'count'))
             .round(2)
             .sort_values('avg_revenue', ascending=False))
by_vendor

,avg_revenue,order_count
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
merch_share = 100 * df.loc[df['category'] == 'Merch', 'revenue'].sum() / df['revenue'].sum()
print(f"Merch share of revenue: {merch_share:.1f}%")

Merch share of revenue: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
baseline_rows = len(df)
baseline_revenue = df['revenue'].sum()

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

print(f"rows after merge: {len(joined)} (expected {baseline_rows})")
print(f"revenue after merge: {joined['revenue'].sum()} (expected {baseline_revenue})")

unmatched = joined[joined['vendor_name'].isna()]
print(f"unmatched vendor_id(s): {unmatched['vendor_id'].unique()}")
print(f"unmatched orders: {len(unmatched)} | revenue at stake: ${unmatched['revenue'].sum():,.2f}")

rows after merge: 400 (expected 400)
revenue after merge: 8520.0 (expected 8520.0)
unmatched vendor_id(s): ['V-18']
unmatched orders: 108 | revenue at stake: $2,349.00


**The unmatched vendor, and what I did about it:** V-18 is missing from the lookup. However, since it accounts for the $2,349 of the total revenue we can't drop it. So I named it unknown vendor V-18 and kept it in the data rather than silently keeping it blank.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor (V-18)')

pivot = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)
pivot.round(2)

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor (V-18),582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

> I apologize but the markdown of the write up is being weird and not producing the clean and complete final product, but if you look at the source the complete response is there.

---
A) Food drives half the business at $4,293. So 50.4% of revenue, so keeping that line staffed and stocked matters most, but RainGear is a distant last at only $901.50 (10.6%), which vendors could treat as an upsell opportunity on rainy game days rather than a core offering. V-01 has the best average order value at $22.60 across a solid 94 orders, suggesting whatever they're doing (pricing mix, upselling) is worth other vendors studying.



---



B) Q3 is the least trustworthy on its own, while 93–108 orders per vendor is a reasonable sample, the gap between the top ($22.60) and bottom ($20.31) average is only about 11%, which is narrow enough that it could partly reflect the random category/price mix each vendor happened to get rather than a real difference in vendor performance. Q5/Q6 also carry a real caveat: "Unknown vendor (V-18)" is doing a lot of work. It's actually the single largest revenue source in the pivot table ($2,349).